# Dynamic Programming - Deterministic Grid World Policy Evaluation 
In the example below we abuse the notion of a transition function $p(s',r|s,a)$. In general, for the case of a $4\times 4$ grid world with four actions $\lbrace 'up', 'down', 'left', 'right'\rbrace$ and a reward of $r = -1$ for all transitions we should end up with $p(s',r|s,a)$ having a shape of $(16, 1, 16, 4)$. However, since we are dealing with a deterministic environment s.t. $p(s',r|s,a)\in \lbrace 0,  1\rbrace$ $\forall s,s'\in S, a\in A$, we will instead of setting the transition function to be a probablity distribution over its dimensions, we set it similar to a `step()` function. Essentially baking the resulted constraints of the transition function into the step function which determines the next state (and reward) of the environment given the action which is taken in the current state.

With this structure in hand we can ignore the internal sum over next states and rewards in the update rule (there is a single possible transition in each state action pair) which will make the algorithm simpler, meanning $\sum_{s',r}p(s',r|s,a) \longrightarrow 1$. In addition, we take $\pi(a|s) = 0.25$ because $\vert A(s)\vert = 4, \forall s\in S$. 

Therefore, for each $s',s,a$ the update rule becomes 

$$v(s) = \sum_{a\in\lbrace up,\,down,\,left,\,right\rbrace}0.25*(-1+v(s'))$$

where $s'=step(s, a)$.

In [1]:
import numpy as np
import copy as cp


class SquareGridWorld:
    """
    Square Grid World
    Barto and Sutton: Chapter 4: Dynamic Programming - Example 4.1
    """
    def __init__(self, size, terminal_states):
        self.size = size
        self.final_state = size ** 2
        self.state_list = np.arange(self.final_state)
        self.state_space = self.state_list.reshape(size, size)
        self.action_space = {'up': (-1, 0), 'down': (1, 0), 'right': (0, 1), 'left': (0, -1)}
        for state in terminal_states:
            if state < 0 or state > self.final_state:
                raise ValueError
        
        self.terminal_states = terminal_states

    def transition_function(self, state, action):
    
        # if in terminal state, do nothing
        if state in self.terminal_states:
            return state, 0, True
        
        # get state grid coordinates 
        i, j = np.unravel_index(state, (self.size, self.size))
    
        # get action grid step
        i_step, j_step = self.action_space[action]
    
        # compute new state (verify it is within bounds)
        i_new = np.clip(i + i_step, 0, self.size - 1)
        j_new = np.clip(j + j_step, 0, self.size - 1)
        new_state = self.state_space[i_new, j_new]
    
        return new_state, -1, False

In [2]:
def print_pretty_matrix(matrix, k=None):
    if k is not None:
        print(f"\nk = {k}:")
    size_i, size_j = matrix.shape
    for i in range(size_i):
        line = f"["
        for j in range(size_j):
            line += f"{matrix[i, j]:5.1f}, "
        line = line[:-2]
        line += f"]"
        print(line)

print_pretty_matrix(np.arange(16).reshape(4, 4))

[  0.0,   1.0,   2.0,   3.0]
[  4.0,   5.0,   6.0,   7.0]
[  8.0,   9.0,  10.0,  11.0]
[ 12.0,  13.0,  14.0,  15.0]


![policy_evaluation.png](../assets/policy_evaluation.png)

In [3]:
# setting some parameters
size = 4
terminal_states = [0, size ** 2 - 1]
gamma = 1.0
theta = 1e-6
max_steps = 1001

In [4]:
# init grid world
grid_world = SquareGridWorld(size, terminal_states)

# init value function
value_function = np.zeros_like(grid_world.state_space, dtype=float)
print_pretty_matrix(value_function, 0)

# iterative policy evaluation
for k in range(1, max_steps):
    delta = 0
    prev_value_function = cp.deepcopy(value_function)
    for state in grid_world.state_list:
        i, j = np.unravel_index(state, (size, size))
        new_value = 0

        # average over the action space
        for action in grid_world.action_space.keys():

            # get transition
            new_state, reward, _ = grid_world.transition_function(state, action)
            new_i, new_j = np.unravel_index(new_state, (size, size))

            # compute update rule (Bellman equation)
            policy = 1 / len(grid_world.action_space.keys())
            new_value += policy * (reward + gamma * prev_value_function[new_i, new_j])

        # update value function
        value_function[i, j] = new_value
        delta = np.max([delta, np.abs(new_value - prev_value_function[i, j])])
    
    if k in [1, 2, 3, 10]:
        print_pretty_matrix(value_function, k)
    
    # check convergence
    if delta < theta:
        break

print_pretty_matrix(value_function, k)


k = 0:
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]

k = 1:
[  0.0,  -1.0,  -1.0,  -1.0]
[ -1.0,  -1.0,  -1.0,  -1.0]
[ -1.0,  -1.0,  -1.0,  -1.0]
[ -1.0,  -1.0,  -1.0,   0.0]

k = 2:
[  0.0,  -1.8,  -2.0,  -2.0]
[ -1.8,  -2.0,  -2.0,  -2.0]
[ -2.0,  -2.0,  -2.0,  -1.8]
[ -2.0,  -2.0,  -1.8,   0.0]

k = 3:
[  0.0,  -2.4,  -2.9,  -3.0]
[ -2.4,  -2.9,  -3.0,  -2.9]
[ -2.9,  -3.0,  -2.9,  -2.4]
[ -3.0,  -2.9,  -2.4,   0.0]

k = 10:
[  0.0,  -6.1,  -8.4,  -9.0]
[ -6.1,  -7.7,  -8.4,  -8.4]
[ -8.4,  -8.4,  -7.7,  -6.1]
[ -9.0,  -8.4,  -6.1,   0.0]

k = 258:
[  0.0, -14.0, -20.0, -22.0]
[-14.0, -18.0, -20.0, -20.0]
[-20.0, -20.0, -18.0, -14.0]
[-22.0, -20.0, -14.0,   0.0]


Once having the Value function, computing the Action-Value function is a single step process.

In [5]:
# init grid world
grid_world = SquareGridWorld(size, terminal_states)

# init action value function
action_value_function = {action: np.zeros_like(grid_world.state_space, dtype=float) for action in grid_world.action_space}

for action in grid_world.action_space.keys():
    print(f"k = {0} - ('{action}'):")
    print_pretty_matrix(action_value_function[action])

# iterative policy evaluation
for k in range(1, 2):
    print("----------------------------")
    prev_action_value_function = cp.deepcopy(action_value_function)
    for state in grid_world.state_list:
        i, j = np.unravel_index(state, grid_world.state_space.shape)

        for action in grid_world.action_space.keys():
            # get transition
            new_state, reward, _ = grid_world.transition_function(state, action)
            new_i, new_j = np.unravel_index(new_state, grid_world.state_space.shape)
    
            # compute update rule (Bellman equation)
            new_action_value = reward + gamma * value_function[new_i, new_j]

            # update action value function
            action_value_function[action][i, j] = new_action_value


for action in grid_world.action_space:
    print(f"\nk = {k} - ('{action}'):")
    print_pretty_matrix(action_value_function[action])

k = 0 - ('up'):
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
k = 0 - ('down'):
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
k = 0 - ('right'):
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
k = 0 - ('left'):
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
[  0.0,   0.0,   0.0,   0.0]
----------------------------

k = 1 - ('up'):
[  0.0, -15.0, -21.0, -23.0]
[ -1.0, -15.0, -21.0, -23.0]
[-15.0, -19.0, -21.0, -21.0]
[-21.0, -21.0, -19.0,   0.0]

k = 1 - ('down'):
[  0.0, -19.0, -21.0, -21.0]
[-21.0, -21.0, -19.0, -15.0]
[-23.0, -21.0, -15.0,  -1.0]
[-23.0, -21.0, -15.0,   0.0]

k = 1 - ('right'):
[  0.0, -21.0, -23.0, -23.0]
[-19.0, -21.0, -21.0, -21.0]
[-21.0, -19.0, -15.0, -15.0]
[-21.0, -15.0,  -1.0,   0.0]

k = 1 - ('left'):
[  0.0,  -1.0

Lets verify we got the same Value function.

In [6]:
varified_value_function = np.zeros_like(value_function)
for state in grid_world.state_list:
    i, j = np.unravel_index(state, grid_world.state_space.shape)
    policy = 1 / len(grid_world.action_space.keys())
    for action in grid_world.action_space.keys():
        varified_value_function[i, j] += policy * action_value_function[action][i, j]

print("Varified value function:")
print_pretty_matrix(varified_value_function)
if np.all(np.isclose(value_function, varified_value_function)):
    print("\nvalue functions are identical")
else:
    print("\nvalue functions are different")

Varified value function:
[  0.0, -14.0, -20.0, -22.0]
[-14.0, -18.0, -20.0, -20.0]
[-20.0, -20.0, -18.0, -14.0]
[-22.0, -20.0, -14.0,   0.0]

value functions are identical
